# INTR on Oxford-IIIT Pet — Kaggle training runner

**Before running:** in the right-hand panel set **Accelerator = GPU T4 x1** (or P100) and **Internet = On**.

**How to use**
1. Set the options in the next cell.
2. **Timing test first:** `TIMING = True`, then *Run All* (interactive). It takes about 10 minutes and prints the suggested number of epochs. Send the printed JSON to the team.
3. **Full run:** set `TIMING = False` and `EPOCHS` to the suggested value, then **Save Version → Save & Run All (Commit)**. It runs in the background (laptop can be closed). Results appear in the version's **Output** tab under `outputs/<RUN>/`.
4. **If a run stops with `PAUSED`** (time guard before Kaggle's 12 h limit): open a new notebook version, *Add Input → Notebook Output* of the paused version, set `RESUME_FROM` to its `outputs/<RUN>` folder, and commit again. Training continues from the last saved epoch.

Runs: `R1` = reproduction (8 heads, shared vector) · `R2` = 1-head decoder cross-attention · `FC` = INTR-FC (one vector per class) · `R3` = ResNet-50 baseline (about 30-60 min).

In [ ]:
# ===================== OPTIONS =====================
RUN = "R1"            # "R1" | "R2" | "FC" | "R3" (ResNet-50 baseline)
TIMING = True         # True = speed test only; False = full training
EPOCHS = 100          # INTR runs: 100 (lr drop at 80% automatically). R3 uses its own default (30) - leave as is
SEED = 42

# Where the code comes from (pick one):
GITHUB_REPO = "https://github.com/TalalImran07/intr-ml-project.git"   # private repo: add a Kaggle Secret named GITHUB_TOKEN
CODE_DATASET = "/kaggle/input/intr-oxford-pet-code"                    # OR upload the code zip as a Kaggle dataset with this name

RESUME_FROM = None    # e.g. "/kaggle/input/<previous-version-output>/outputs/R1"
# ===================================================
VARIANT_FLAGS = {"R1": "", "R2": "--dec_heads 1", "FC": "--head fc", "R3": ""}[RUN]
SCRIPT = "train_resnet.py" if RUN == "R3" else "train.py"

In [ ]:
# Get the code into /kaggle/working/code
import os, shutil, subprocess
CODE = "/kaggle/working/code"
if os.path.exists(CODE):
    shutil.rmtree(CODE)
src = None
if os.path.isdir(CODE_DATASET):
    # a zip uploaded as a dataset is unpacked by Kaggle; find the folder that holds train.py
    for dirpath, _, files in os.walk(CODE_DATASET):
        if "train.py" in files:
            src = dirpath; break
if src:
    shutil.copytree(src, CODE)
    print("code copied from dataset:", src)
else:
    url = GITHUB_REPO
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = url.replace("https://", f"https://{tok}@")
    except Exception:
        pass   # public repo or no secret set
    subprocess.run(["git", "clone", "--depth", "1", url, CODE], check=True)
    print(subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%h %cd %s"],
                         capture_output=True, text=True).stdout)
os.chdir(CODE)
print(sorted(os.listdir(CODE)))

In [ ]:
import torch, torchvision, platform
print("python", platform.python_version(), "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU accelerator!")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Download Oxford-IIIT Pet (~800 MB) into /tmp (not saved as notebook output) and build the split file
!python prepare_data.py --root /tmp/data

In [ ]:
OUT = "/kaggle/working/outputs"
cmd = f"python {SCRIPT} --run_name {RUN} --data_root /tmp/data --out_root {OUT} --seed {SEED} {VARIANT_FLAGS}"
if RUN != "R3":
    cmd += f" --epochs {EPOCHS}"
if TIMING:
    cmd += " --timing 50"
if RESUME_FROM and RUN != "R3":
    cmd += f" --resume_dir {RESUME_FROM}"
print(cmd)
!{cmd}

In [ ]:
# Summary of this session
import json, pathlib, pandas as pd
d = pathlib.Path(OUT) / RUN
for f in ["timing.json", "metrics.json", "STATUS.txt"]:
    if (d / f).exists():
        print(f"--- {f}\n" + (d / f).read_text())
if (d / "log.csv").exists():
    display(pd.read_csv(d / "log.csv").tail(15))
print(sorted(p.name for p in d.iterdir()) if d.exists() else "no output folder")